# NeuroTrace
**U-Net Brain Tumor Segmentation** — CS 571

Predicts pixel-level tumor masks from LGG brain MRI images using a U-Net trained with combined BCE + Dice loss.

Run cells top to bottom. Make sure the Colab runtime is set to **GPU** (Runtime → Change runtime type → T4 GPU).

## 1. Install dependencies

In [ ]:
!pip install -q albumentations opencv-python-headless tqdm scikit-learn

## 2. Upload / mount the dataset

Download **LGG Brain MRI Segmentation** from Kaggle:  
https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation

Then either:
- **Option A** — Mount Google Drive and point `DATA_DIR` at the extracted folder, or  
- **Option B** — Upload the zip directly and unzip it here.

In [ ]:
import os, glob
from google.colab import drive

drive.mount('/content/drive')

ARCHIVE = '/content/drive/MyDrive/NeuroTrace/archive.zip'
EXTRACT_TO = '/content/dataset/'
os.makedirs(EXTRACT_TO, exist_ok=True)

!apt-get install -q p7zip-full
!7z x "$ARCHIVE" -o"$EXTRACT_TO" -y

# Find kaggle_3m wherever it landed
matches = glob.glob(f'{EXTRACT_TO}**/kaggle_3m', recursive=True)
if not matches:
    contents = os.listdir(EXTRACT_TO)
    raise FileNotFoundError(f'kaggle_3m not found. Contents: {contents}')

DATA_DIR = matches[0]
print(f'DATA_DIR = {DATA_DIR}')

## 3. Clone / upload the project files

If you pushed this repo to GitHub, clone it. Otherwise upload `data.py`, `model.py`, `train.py`, and `eval.py` manually.

In [ ]:
# Clone from GitHub (replace with your repo URL)
# !git clone https://github.com/<your-username>/NeuroTrace.git
# %cd NeuroTrace

# Or just verify the files are present in the current directory
import os
for f in ['data.py', 'model.py', 'train.py', 'eval.py']:
    status = '✓' if os.path.exists(f) else '✗  MISSING'
    print(f"{status}  {f}")

## 4. Imports & device

In [ ]:
import torch
from data  import build_dataloaders
from model import UNet, count_parameters
from train import train
from eval  import evaluate, visualize_predictions, plot_training_history

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 5. Hyperparameters

In [ ]:
IMAGE_SIZE   = 256
BATCH_SIZE   = 32
EPOCHS       = 60
LR           = 1e-4
WEIGHT_DECAY = 1e-5
CHECKPOINT   = 'best_model.pth'

## 6. Build dataloaders

In [ ]:
train_loader, val_loader, test_loader = build_dataloaders(
    DATA_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

## 7. Visualize a sample batch

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

images, masks = next(iter(train_loader))

mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(4):
    img = images[i].numpy().transpose(1, 2, 0)
    img = np.clip(std * img + mean, 0, 1)
    axes[0, i].imshow(img)
    axes[0, i].set_title('MRI')
    axes[0, i].axis('off')
    axes[1, i].imshow(masks[i].squeeze(), cmap='gray')
    axes[1, i].set_title('Mask')
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()

## 8. Build the U-Net model

In [ ]:
model = UNet(in_channels=3, out_channels=1)
count_parameters(model)

## 9. Train

In [ ]:
history = train(
    model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    checkpoint_path=CHECKPOINT,
    device=DEVICE,
    use_amp=True,
)

## 10. Plot training curves

In [ ]:
plot_training_history(history)

## 11. Evaluate on the test set

In [ ]:
# Load the best checkpoint before evaluating
model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE))

results = evaluate(model, test_loader, DEVICE)

## 12. Visualize predictions

In [ ]:
visualize_predictions(model, test_loader, DEVICE, n_samples=4)